In [71]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PJM_PRICE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pjm"
    / "da_hrl_lmps_PJM_PS.csv"
)

PJM_LOAD_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pjm"
    / "hrl_load_metered_PJM_PS.csv"
)

NYISO_PRICE_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nyiso"
    / "nyiso_hudson_valley_jan2025_LMP_DATA.csv"
)

NYISO_LOAD_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nyiso"
    / "nyiso_hudson_valley_jan2025palIntegrated_HV_loaddata.csv"
)

NOAA_WEATHER_FILE_STEWART =(
    PROJECT_ROOT
    /"data"
    /"raw"
    /"raw"
    /""

)
NOAA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "noaa"
)

NEWARK_WEATHER_FILE = (
    NOAA_DIR
    / "WeatherData Jan25 Newark.csv"
)

STEWART_WEATHER_FILE = (
    NOAA_DIR
    / "WeatherData Jan25 Stewart.csv"
)

newark_weather_raw = pd.read_csv(
    NEWARK_WEATHER_FILE,
    low_memory=False,
)

stewart_weather_raw = pd.read_csv(
    STEWART_WEATHER_FILE,
    low_memory=False,
)

pjm_price_raw = pd.read_csv(PJM_PRICE_FILE)
pjm_load_raw = pd.read_csv(PJM_LOAD_FILE)
nyiso_price_raw =pd.read_csv(NYISO_PRICE_FILE)
nyiso_load_raw = pd.read_csv(NYISO_LOAD_FILE)


print("PJM price:", pjm_price_raw.shape)
print("PJM load:", pjm_load_raw.shape)
print("NYISO price:", nyiso_price_raw.shape)
print("NYISO load:", nyiso_load_raw.shape)
print("Newark file exists:", NEWARK_WEATHER_FILE.exists())
print("Stewart file exists:", STEWART_WEATHER_FILE.exists())

print("Newark weather shape:", newark_weather_raw.shape)
print("Stewart weather shape:", stewart_weather_raw.shape)

print(nyiso_price_raw.columns.tolist())
print(nyiso_load_raw.columns.tolist())

PJM price: (744, 14)
PJM load: (744, 8)
NYISO price: (744, 7)
NYISO load: (744, 6)
Newark file exists: True
Stewart file exists: True
Newark weather shape: (12970, 125)
Stewart weather shape: (9081, 125)
['Time Stamp', 'Name', 'PTID', 'LBMP ($/MWHr)', 'Marginal Cost Losses ($/MWHr)', 'Marginal Cost Congestion ($/MWHr)', 'source_file']
['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Integrated Load', 'source_file']


In [72]:
print(pjm_price_raw.shape)
pjm_price_raw.head()

(744, 14)


,datetime_beginning_utc,datetime_beginning_ept,pnode_id,pnode_name,voltage,equipment,type,zone,system_energy_price_da,total_lmp_da,congestion_price_da,marginal_loss_price_da,row_is_current,version_nbr
0,1/1/2025 5:00:00 AM,1/1/2025 12:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,21.26,21.981200,0.153437,0.567763,True,1
1,1/1/2025 6:00:00 AM,1/1/2025 1:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.96,21.397873,-0.012770,0.450643,True,1
2,1/1/2025 7:00:00 AM,1/1/2025 2:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.42,20.641678,-0.266470,0.488148,True,1
3,1/1/2025 8:00:00 AM,1/1/2025 3:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.45,20.696320,-0.283920,0.530240,True,1
4,1/1/2025 9:00:00 AM,1/1/2025 4:00:00 AM,51301,PSEG,NaN,NaN,ZONE,NaN,20.46,20.649481,-0.400733,0.590214,True,1


In [73]:
pjm_price = pjm_price_raw.copy()

pjm_price["timestamp_local"] = pd.to_datetime(
    pjm_price["datetime_beginning_ept"],
    format="%m/%d/%Y %I:%M:%S %p",
)
"pjm_price_raw" in globals()

True

In [74]:
pjm_price = pjm_price[
    [
        "timestamp_local",
        "pnode_id",
        "pnode_name",
        "total_lmp_da",
        "system_energy_price_da",
        "congestion_price_da",
        "marginal_loss_price_da",
    ]
].rename(
    columns={
        "pnode_id": "location_id",
        "pnode_name": "location",
        "total_lmp_da": "day_ahead_price_usd_mwh",
        "system_energy_price_da": "energy_component_usd_mwh",
        "congestion_price_da": "congestion_component_usd_mwh",
        "marginal_loss_price_da": "loss_component_usd_mwh",
    }
)

In [75]:
print(pjm_price.shape)
print(pjm_price["timestamp_local"].min())
print(pjm_price["timestamp_local"].max())
print("Duplicate hours:", pjm_price["timestamp_local"].duplicated().sum())
print("Missing prices:", pjm_price["day_ahead_price_usd_mwh"].isna().sum())

(744, 7)
2025-01-01 00:00:00
2025-01-31 23:00:00
Duplicate hours: 0
Missing prices: 0


In [76]:
pjm_load = pjm_load_raw.copy()

pjm_load["timestamp_local"] = pd.to_datetime(
    pjm_load["datetime_beginning_ept"],
    format="%m/%d/%Y %I:%M:%S %p",
)

pjm_load = pjm_load[
    ["timestamp_local", "mw", "is_verified"]
].rename(
    columns={
        "mw": "actual_load_mw",
    }
)

In [77]:
assert len(pjm_load) == 744
assert pjm_load["timestamp_local"].is_unique
assert pjm_load["actual_load_mw"].notna().all()

In [78]:
nyiso_price = nyiso_price_raw.copy()

nyiso_price["timestamp_local"] = pd.to_datetime(
    nyiso_price["Time Stamp"]
)

nyiso_price = nyiso_price[
    [
        "timestamp_local",
        "PTID",
        "Name",
        "LBMP ($/MWHr)",
        "Marginal Cost Losses ($/MWHr)",
        "Marginal Cost Congestion ($/MWHr)",
    ]
].rename(
    columns={
        "PTID": "location_id",
        "Name": "location",
        "LBMP ($/MWHr)": "day_ahead_price_usd_mwh",
        "Marginal Cost Losses ($/MWHr)": "loss_component_usd_mwh",
        "Marginal Cost Congestion ($/MWHr)":
            "congestion_component_usd_mwh",
    }
)

In [79]:
nyiso_price["energy_component_usd_mwh"] = (
    nyiso_price["day_ahead_price_usd_mwh"]
    - nyiso_price["loss_component_usd_mwh"]
    - nyiso_price["congestion_component_usd_mwh"]
)

In [80]:
nyiso_load = nyiso_load_raw.copy()

nyiso_load["timestamp_local"] = pd.to_datetime(
    nyiso_load["Time Stamp"]
)

nyiso_load = nyiso_load[
    ["timestamp_local", "Integrated Load"]
].rename(
    columns={
        "Integrated Load": "actual_load_mw",
    }
)

In [81]:
assert len(nyiso_price) == 744
assert len(nyiso_load) == 744
assert nyiso_price["timestamp_local"].is_unique
assert nyiso_load["timestamp_local"].is_unique

In [82]:
JANUARY_HOURS = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-01-31 23:00:00",
    freq="h",
)

In [83]:
def clean_weather(
    raw_weather: pd.DataFrame,
    station_code: str,
) -> pd.DataFrame:
    weather = raw_weather.copy()

    weather["observed_at"] = pd.to_datetime(
        weather["DATE"],
        errors="coerce",
    )

    weather = weather[
        weather["observed_at"].between(
            "2025-01-01",
            "2025-01-31 23:59:59",
        )
        & weather["REPORT_TYPE"].isin(
            ["FM-12", "FM-15", "FM-16"]
        )
    ].copy()

    weather["temperature_c"] = pd.to_numeric(
        weather["HourlyDryBulbTemperature"],
        errors="coerce",
    )

    weather["dew_point_c"] = pd.to_numeric(
        weather["HourlyDewPointTemperature"],
        errors="coerce",
    )

    weather["relative_humidity_pct"] = pd.to_numeric(
        weather["HourlyRelativeHumidity"],
        errors="coerce",
    )

    weather["wind_speed_mps"] = pd.to_numeric(
        weather["HourlyWindSpeed"],
        errors="coerce",
    )

    weather.loc[
        ~weather["temperature_c"].between(-50, 50),
        "temperature_c",
    ] = pd.NA

    weather.loc[
        ~weather["dew_point_c"].between(-60, 40),
        "dew_point_c",
    ] = pd.NA

    weather.loc[
        ~weather["relative_humidity_pct"].between(0, 100),
        "relative_humidity_pct",
    ] = pd.NA

    weather.loc[
        ~weather["wind_speed_mps"].between(0, 75),
        "wind_speed_mps",
    ] = pd.NA

    weather["timestamp_local"] = (
        weather["observed_at"].dt.floor("h")
    )

    value_columns = [
        "temperature_c",
        "dew_point_c",
        "relative_humidity_pct",
        "wind_speed_mps",
    ]

    hourly = (
        weather.groupby("timestamp_local")[value_columns]
        .median()
        .reindex(JANUARY_HOURS)
    )

    missing_before = hourly[value_columns].isna()

    hourly[value_columns] = hourly[value_columns].interpolate(
        method="time",
        limit_direction="both",
    )

    hourly["weather_imputed"] = missing_before.any(axis=1)
    hourly["weather_station"] = station_code
    hourly.index.name = "timestamp_local"

    return hourly.reset_index()

In [84]:
newark_weather = clean_weather(
    newark_weather_raw,
    station_code="USW00014734",
)

stewart_weather = clean_weather(
    stewart_weather_raw,
    station_code="USW00014714",
)

In [85]:
datasets = {
    "PJM price": pjm_price_raw,
    "PJM load": pjm_load_raw,
    "NYISO price": nyiso_price_raw,
    "NYISO load": nyiso_load_raw,
    "Newark weather": newark_weather_raw,
    "Stewart weather": stewart_weather_raw,
}

for name, df in datasets.items():
    print(f"{name:20} rows={df.shape[0]:5} columns={df.shape[1]:3}")

PJM price            rows=  744 columns= 14
PJM load             rows=  744 columns=  8
NYISO price          rows=  744 columns=  7
NYISO load           rows=  744 columns=  6
Newark weather       rows=12970 columns=125
Stewart weather      rows= 9081 columns=125


In [86]:
assert pjm_price_raw.shape == (744, 14)
assert pjm_load_raw.shape == (744, 8)
assert nyiso_price_raw.shape == (744, 7)
assert nyiso_load_raw.shape == (744, 6)

print("All four electricity files passed the row-count checks.")

All four electricity files passed the row-count checks.


In [87]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())


PJM price
['datetime_beginning_utc', 'datetime_beginning_ept', 'pnode_id', 'pnode_name', 'voltage', 'equipment', 'type', 'zone', 'system_energy_price_da', 'total_lmp_da', 'congestion_price_da', 'marginal_loss_price_da', 'row_is_current', 'version_nbr']

PJM load
['datetime_beginning_utc', 'datetime_beginning_ept', 'nerc_region', 'mkt_region', 'zone', 'load_area', 'mw', 'is_verified']

NYISO price
['Time Stamp', 'Name', 'PTID', 'LBMP ($/MWHr)', 'Marginal Cost Losses ($/MWHr)', 'Marginal Cost Congestion ($/MWHr)', 'source_file']

NYISO load
['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Integrated Load', 'source_file']

Newark weather
['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'REPORT_TYPE', 'SOURCE', 'HourlyAltimeterSetting', 'HourlyDewPointTemperature', 'HourlyDryBulbTemperature', 'HourlyPrecipitation', 'HourlyPresentWeatherType', 'HourlyPressureChange', 'HourlyPressureTendency', 'HourlyRelativeHumidity', 'HourlySkyConditions', 'HourlySeaLevelPressure', 'HourlySta

In [88]:
pjm_price = pjm_price_raw.copy()

pjm_price["timestamp_local"] = pd.to_datetime(
    pjm_price["datetime_beginning_ept"],
    format="%m/%d/%Y %I:%M:%S %p",
)

pjm_price = pjm_price[
    [
        "timestamp_local",
        "pnode_id",
        "pnode_name",
        "total_lmp_da",
        "system_energy_price_da",
        "congestion_price_da",
        "marginal_loss_price_da",
    ]
].rename(
    columns={
        "pnode_id": "location_id",
        "pnode_name": "location",
        "total_lmp_da": "day_ahead_price_usd_mwh",
        "system_energy_price_da": "energy_component_usd_mwh",
        "congestion_price_da": "congestion_component_usd_mwh",
        "marginal_loss_price_da": "loss_component_usd_mwh",
    }
)

In [89]:
print("Shape:", pjm_price.shape)
print("First hour:", pjm_price["timestamp_local"].min())
print("Last hour:", pjm_price["timestamp_local"].max())
print("Duplicate hours:", pjm_price["timestamp_local"].duplicated().sum())
print("Missing prices:", pjm_price["day_ahead_price_usd_mwh"].isna().sum())

pjm_price.head()

Shape: (744, 7)
First hour: 2025-01-01 00:00:00
Last hour: 2025-01-31 23:00:00
Duplicate hours: 0
Missing prices: 0


,timestamp_local,location_id,location,day_ahead_price_usd_mwh,energy_component_usd_mwh,congestion_component_usd_mwh,loss_component_usd_mwh
0,2025-01-01 00:00:00,51301,PSEG,21.981200,21.26,0.153437,0.567763
1,2025-01-01 01:00:00,51301,PSEG,21.397873,20.96,-0.012770,0.450643
2,2025-01-01 02:00:00,51301,PSEG,20.641678,20.42,-0.266470,0.488148
3,2025-01-01 03:00:00,51301,PSEG,20.696320,20.45,-0.283920,0.530240
4,2025-01-01 04:00:00,51301,PSEG,20.649481,20.46,-0.400733,0.590214
